# 零樣本分類：不用任何標註資料，只靠「標籤的文字描述」
---

## 這份程式要做什麼

你在程式開頭寫的註解，已經把核心講完了：

> 沒有已標記資料，只有標籤本身；模型決定輸入應該與候選標籤產生相關連方式。

也就是：**我們完全沒有用到任何「影評 → 正面或負面」的標註資料，連訓練都不需要。** 只做一件事：

```
把「影評」和「標籤的文字描述」都變成向量，放在同一個空間裡，
看每句影評和哪個標籤的向量比較近，就預測那個標籤。
```

```
標籤描述：   "A negative review"  →  向量 L0
            "A positive review"  →  向量 L1

一句影評：   "an unforgettable and moving film"  →  向量 R

比較 R 與 L0 的距離、R 與 L1 的距離   →   較近的那個就是預測
```

## 這是第四種做法

| 做法 | 需要標註資料 | 訓練 |
|---|---|---|
| ① 微調 BERT | 多 | 大 |
| ② 凍結 Embedding + 邏輯回歸 | 多 | 小 |
| ③ 現成的任務專用模型 | 不需要 | 無（但要有人針對情緒訓練過模型） |
| **④ 零樣本分類（這份）** | **完全不需要** | **完全沒有** |

③ 和 ④ 的差別：③ 用的是「別人針對情緒任務訓練好的模型」；④ 用的是**通用的句向量模型**，它從來沒學過「情緒分類」這件事，只是靠「語意相近的句子，向量也相近」這個通用能力。

## 為什麼這個做法可行

`all-mpnet-base-v2` 是在大量句子對上用對比學習訓練的：語意相近的句子，向量靠近。所以：

- 「A positive review」這個句子的向量，會靠近「正面影評」的向量
- 「A negative review」這個句子的向量，會靠近「負面影評」的向量

這是我依句向量模型的訓練方式做的說明，不是這個模型的保證，實際效果要看實驗結果。

## 零樣本是什麼意思

| 名稱 | 意思 |
|---|---|
| **零樣本（zero-shot）** | 對這個任務，**一筆標註範例都沒給**，只告訴模型有哪些類別 |
| 少樣本（few-shot） | 只給少量範例（例如 SetFit） |
| 監督式 | 給大量標註範例 |

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入影評資料集 |
| `SentenceTransformer` | 載入 Embedding 模型，把文字轉成向量 |
| `cosine_similarity`（scikit-learn） | 計算**餘弦相似度**，比較向量有多接近 |
| `classification_report` | 產生分類評估報告 |
| `numpy.shape` | 取得陣列的形狀（後面用來印出相似度矩陣的大小） |

這份和上一份「凍結 Embedding + 邏輯回歸」的差別在於：**沒有 `LogisticRegression`**，因為不需要訓練任何分類器，取而代之的是 `cosine_similarity`。

In [ ]:
#====Zero-Shot Classification====
#沒有已標記資料 只有標籤本身 模型決定輸入應該與候選標籤產生相關連方式

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report
from numpy import shape

## 步驟 1：定義評估函式

```python
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)
```

和前兩份完全相同的評估函式。**四種做法用同一個函式評估，結果才能公平比較。**

| 部分 | 說明 |
|---|---|
| `y_true` | 真正的答案（資料集的標籤） |
| `y_pred` | 模型的預測 |
| `target_names` | 把 0、1 換成名稱，順序必須對應標籤編號（0 = 負面、1 = 正面） |

In [ ]:
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)

## 步驟 2：載入資料

```python
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
```

| 欄位 | 說明 |
|---|---|
| `text` | 一句影評 |
| `label` | `0` = 負面、`1` = 正面 |

這份程式**只用測試集**（約 1066 筆，正負各半，我印象中如此）。**完全不用訓練集**，因為不訓練，也不需要任何標註資料來「學習」，標籤只在最後拿來評估。


In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# print(data)
# print(data["train"][0])  #印出第一筆資料

## 步驟 3：載入 Embedding 模型

```python
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
```

和上一份用的是**同一個模型**，丟一句話進去，吐出一個 768 維的向量（我印象中如此）。

這個模型**沒有被訓練、也沒有被微調**，完全是拿來就用。**這份程式的全部「智慧」，都來自這個預訓練模型的通用語意能力。**

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

## 步驟 4：把「標籤描述」和「影評」都編碼成向量

```python
label_embeddings = model.encode(["A negative review", "A positive review"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
```

### 這是整份程式最關鍵的設計

我們把**兩種不同的東西**放進**同一個向量空間**：

| 變數 | 內容 | 形狀 |
|---|---|---|
| `label_embeddings` | **標籤的文字描述**（兩句話）的向量 | `(2, 768)` |
| `test_embeddings` | **所有測試影評**的向量 | `(測試筆數, 768)`，例如 `(1066, 768)` |

### 標籤描述的順序很重要

```python
["A negative review", "A positive review"]
#     第 0 個              第 1 個
```

第 0 個是負面、第 1 個是正面，**必須和資料集的標籤編號一致**（0 = 負面、1 = 正面）。後面用 `argmax` 得到的是「第幾個標籤」，這個編號會直接拿去和真實標籤比較。順序寫反，預測會整個顛倒，而且不會報錯。

### 為什麼標籤要寫成句子，而不是單字 `"negative"`

句向量模型是針對「**一句話**」訓練的。`"A negative review"` 是一個完整的、語意明確的短句，通常比單一個字 `"negative"` 更適合和「影評這種句子」比較。這是我的推測，實際差異要靠實驗確認（見實驗區）。**標籤描述怎麼寫，會直接影響結果**，這就是零樣本方法的「調校點」。

### 小提醒

`show_progress_bar=True` 用在只有兩個句子的 `label_embeddings` 上，沒有實際用處（兩句話瞬間就編碼完），可以忽略。`test_embeddings` 那一行才是**最慢**的部分，要等一下。

In [ ]:
label_embeddings = model.encode(["A negative review", "A positive review"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

## 步驟 5：計算相似度，並取最像的標籤

```python
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
print(shape(sim_matrix))
y_pred = sim_matrix.argmax(axis=1)
```

### 餘弦相似度

餘弦相似度比較兩個向量的**方向**有多接近，**不看長度**：

| 值 | 意思 |
|---|---|
| 接近 1 | 方向幾乎相同，語意很相近 |
| 接近 0 | 幾乎無關 |
| 接近 -1 | 方向相反 |

### 相似度矩陣

`cosine_similarity(A, B)` 會算出 **A 的每一列和 B 的每一列** 兩兩之間的相似度：

```
test_embeddings  : (測試筆數, 768)
label_embeddings : (2, 768)
sim_matrix       : (測試筆數, 2)     ← 每句影評對「兩個標籤」各有一個相似度
```

`print(shape(sim_matrix))` 印出的就是這個形狀，你的註解寫的 `shape=(test_data_size, label_size)` 就是這個意思。

```
sim_matrix 的某一列：[0.12, 0.31]      ← [對 negative 的相似度, 對 positive 的相似度]
argmax(axis=1)  → 1                   ← 第 1 欄（positive）比較大
```

對每句影評，選出相似度較高的那個標籤的**編號**。這個編號（0 或 1）剛好對應資料集的標籤，所以可以直接當作預測。

### 這個方法的特性

| 特性 | 說明 |
|---|---|
| **只看相對大小** | 只比較「哪個標籤比較近」，不看絕對相似度有多高 |
| **永遠二選一** | 即使兩個標籤的相似度差距很小（例如評論 3、4），還是會選一個，沒有「不確定」這個選項 |
| **沒有閾值、沒有訓練** | 不需要設定任何門檻 |

In [ ]:
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
print(shape(sim_matrix)) #印出相似度矩陣, shape=(test_data_size, label_size)
y_pred = sim_matrix.argmax(axis=1)

## 步驟 6：評估

```python
evaluate_performance(data["test"]["label"], y_pred)
```

**注意參數順序：第一個是真正的答案，第二個是預測。**

### 怎麼讀報告

`precision`（模型說是這一類的，有多少真的是）、`recall`（真正是這一類的，模型找出多少）、`f1-score`（兩者的調和平均）、`support`（該類在測試集的真實筆數）。

### 預期的表現

零樣本通常比有訓練的方法**差一些**，因為模型沒有看過任何「影評 → 情緒」的範例。我印象中這個方法在這個資料集上，F1 大約在 0.78 上下，但這只是印象，沒有確認，請以你跑出來的為準。

比較四種做法時可以這樣想：

| 如果結果是… | 可能的原因 |
|---|---|
| 比有訓練的做法**低** | 零樣本沒有學過任務；標籤描述可能不夠好 |
| 卻已經**不錯** | 通用句向量本身就帶有相當多情緒的資訊 |
| 兩類的 precision / recall 不平衡 | 其中一個標籤描述在向量空間裡「更靠近一般影評」，容易被選中（見實驗區） |

零樣本的價值不在於最高的分數，而在於：**完全不需要標註資料，就能得到一個不錯的起點**。

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

---

# 重點整理

## 一、整份程式一句話

> 把兩個標籤的文字描述（"A negative review"、"A positive review"）和所有測試影評，用同一個句向量模型編碼到同一個空間，計算每句影評與兩個標籤的餘弦相似度，取較近的那個標籤作為預測，全程沒有任何訓練或標註資料。

## 二、整條流程（必須搞懂）

| 步驟 | 程式 | 結果 |
|---|---|---|
| 編碼標籤描述 | `model.encode(["A negative review", "A positive review"])` | 形狀 `(2, 768)` |
| 編碼影評 | `model.encode(data["test"]["text"])` | 形狀 `(測試筆數, 768)` |
| 算相似度 | `cosine_similarity(test_embeddings, label_embeddings)` | 形狀 `(測試筆數, 2)` |
| 選最近的標籤 | `.argmax(axis=1)` | 每句一個 0 或 1 |
| 評估 | `evaluate_performance(...)` | 分類報告 |

## 三、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 零樣本（zero-shot） | 對任務完全沒有標註範例，只告訴模型有哪些類別 |
| 2 | 同一個向量空間 | 影評和標籤描述用同一個模型編碼，才能比較 |
| 3 | 標籤描述 | 把標籤寫成一句話，是這個方法的「調校點」 |
| 4 | 餘弦相似度 | 比較向量的方向，不看長度 |
| 5 | 相似度矩陣 | 每句影評對每個標籤各一個相似度，形狀 `(筆數, 標籤數)` |
| 6 | `argmax(axis=1)` | 沿著欄位方向取最大者，得到每句最近的標籤編號 |
| 7 | 標籤順序 | 描述的順序必須對應資料集的標籤編號 |
| 8 | 強迫二選一 | 永遠會選一個標籤，沒有「不確定」 |
| 9 | 通用能力 | 靠句向量模型的「語意相近 = 向量相近」，不是針對情緒訓練的 |
| 10 | 基準線 | 零成本的最低標準，用來衡量其他做法值不值得 |

## 四、最容易搞混的地方

1. **這份有訓練嗎？**
   完全沒有。沒有訓練模型，也沒有訓練分類器。

2. **這份有用到標註資料嗎？**
   只在最後評估時用到測試集的標籤。過程中沒有任何資料被拿來「學習」。

3. **和上一份「凍結 Embedding + 邏輯回歸」差在哪？**
   上一份用訓練集標籤訓練了一個分類器；這份連分類器都沒有，只比較距離。

4. **和第三種做法（現成情緒模型）差在哪？**
   第三種用的是「專門針對情緒訓練過」的模型；這份用的是通用句向量模型，從沒學過情緒分類。

5. **為什麼要把標籤寫成句子？**
   句向量模型是針對句子訓練的，完整的句子通常比單一個字更適合和影評比較。

6. **`axis=1` 是什麼方向？**
   沿著每一列、在欄位之間比較。矩陣的每一列是一句影評、每一欄是一個標籤。

7. **標籤描述順序寫反會怎樣？**
   預測整個顛倒，分數會變很差，而且不會報錯。

8. **為什麼不管多不確定都要選一個？**
   `argmax` 只看相對大小，沒有門檻。想要「不確定」選項，要自己加閾值（見實驗區）。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| 分數很差、近乎隨機 | 標籤描述不好，或描述的順序和標籤編號不一致 | 確認順序（0 = 負面、1 = 正面）；嘗試不同的描述 |
| 報告裡 negative 和 positive 的分數顛倒 | 標籤描述順序寫反 | 確認 `["A negative review", "A positive review"]` 的順序 |
| 某一類的 recall 特別低、另一類特別高 | 其中一個描述在向量空間裡偏離一般影評 | 換個描述，或多寫幾個描述取平均（見實驗） |
| `sim_matrix` 形狀不是 `(筆數, 2)` | 傳入的矩陣順序反了 | 第一個參數是影評向量，第二個是標籤向量 |
| 編碼非常慢 | 在 CPU 上跑，或資料量大 | 使用 GPU；或先用少量資料測試 |
| `encode` 報型別錯誤 | 傳入的不是一般的 list（新版 `datasets` 取欄位可能得到特殊物件） | 轉成 list：`list(data["test"]["text"])` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **結果對標籤描述非常敏感**
   `"A negative review"` 換成別的說法，分數可能明顯改變。

2. **標籤順序要手動對應標籤編號**
   寫反不會報錯。

3. **沒有信心程度**
   只取 `argmax`，看不出模型對這個預測有多確定。兩個相似度很接近的影評，和差很多的影評，被一視同仁。

4. **`show_progress_bar=True` 對兩個標籤沒有用處**
   不影響結果。

5. **`from numpy import shape`**
   只為了印形狀，直接寫 `sim_matrix.shape` 更簡單，結果相同。

6. **每次都要重新編碼**
   `test_embeddings` 是最慢的一步，結果沒有存起來。

7. **向量空間同時含有「主題」和「情緒」**
   句向量不只編碼情緒，也編碼主題；一部談到「評論」或「糟糕」這類字眼的影評，可能被拉向某個標籤。

8. **只有單次結果**
   沒有比較不同描述寫法的差異。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看相似度矩陣的前幾列
import numpy as np
print(np.round(sim_matrix[:5], 3))             # 每列是 [對 negative, 對 positive]

# 2. 換標籤描述的寫法，比較分數
variants = {
    "baseline":   ["A negative review", "A positive review"],
    "movie":      ["A negative movie review", "A positive movie review"],
    "very":       ["A very negative movie review", "A very positive movie review"],
    "sentence":   ["This movie is terrible.", "This movie is wonderful."],
}
for name, labels in variants.items():
    le = model.encode(labels)
    pred = cosine_similarity(test_embeddings, le).argmax(axis=1)
    print("====", name)
    evaluate_performance(data["test"]["label"], pred)

# 3. 每個標籤用「多個描述」，取向量平均，通常更穩定
neg_texts = ["A negative review", "A terrible movie", "I hated this film"]
pos_texts = ["A positive review", "A wonderful movie", "I loved this film"]
neg_vec = model.encode(neg_texts).mean(axis=0)
pos_vec = model.encode(pos_texts).mean(axis=0)
label_embeddings2 = np.vstack([neg_vec, pos_vec])
pred = cosine_similarity(test_embeddings, label_embeddings2).argmax(axis=1)
evaluate_performance(data["test"]["label"], pred)

# 4. 找出「模稜兩可」的影評：兩個相似度差距很小
margin = np.abs(sim_matrix[:, 1] - sim_matrix[:, 0])
uncertain = np.argsort(margin)[:5]
for i in uncertain:
    print(round(float(margin[i]), 4), data["test"]["text"][i])

# 5. 加上「不確定」選項：差距太小就不預測
threshold = 0.02
confident = margin > threshold
print("有信心的比例:", confident.mean())

# 6. 少樣本升級：用訓練集每類的向量平均當「標籤向量」（不需要訓練分類器）
train_emb = model.encode(data["train"]["text"])
train_y = np.array(data["train"]["label"])
centroids = np.vstack([train_emb[train_y == 0].mean(axis=0),
                       train_emb[train_y == 1].mean(axis=0)])
pred = cosine_similarity(test_embeddings, centroids).argmax(axis=1)
evaluate_performance(data["test"]["label"], pred)

# 7. 換成別的句向量模型，看零樣本表現是否改變
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 8. 擴充成三類（加上中性），觀察模型是否能分辨
# ["A negative review", "A neutral review", "A positive review"]
```

## 八、四種影評情緒分類做法的對照

| | ① 微調 BERT | ② 凍結 Embedding + 邏輯回歸 | ③ 現成任務模型 | **④ 零樣本（這份）** |
|---|---|---|---|---|
| 需要標註資料 | 多 | 多 | 不需要 | **完全不需要** |
| 訓練 | 大 | 小 | 無 | **無** |
| 需要針對任務訓練好的模型 | 否（自己訓練） | 否 | **是** | **否** |
| 可調校的地方 | 超參數、資料 | 分類器 | 幾乎沒有 | **標籤描述** |
| 實作複雜度 | 高 | 低 | 最低 | **最低** |
| 效果（一般） | 最好 | 不錯 | 視領域是否匹配 | 較低，但是零成本 |
| 能擴充到新類別 | 要重新訓練 | 要重新訓練 | 受限於模型的類別 | **只要多寫一個描述** |

最後一列是零樣本最大的優勢：**想新增一個類別，只要多寫一句話**，不需要重新訓練。

## 九、怎麼選

| 你的情況 | 建議 |
|---|---|
| 完全沒有標註資料，想快速試試看 | **這份：零樣本** |
| 類別經常變動、要快速加新類別 | **零樣本** |
| 有少量標註資料（幾十到幾百筆） | 凍結 Embedding + 邏輯回歸，或 SetFit |
| 找得到針對你任務與領域的現成模型 | 現成任務模型 |
| 標註資料充足、追求最佳效果 | 微調整個模型 |

## 十、一句話記住

> **零樣本 = 把標籤寫成一句話，和資料放進同一個向量空間，看誰離誰近。** 沒有訓練、沒有標註資料，只靠句向量模型的通用語意能力，就能做出一個基準線，而標籤描述怎麼寫，就是你唯一可以調整的旋鈕。